# MAP survey

This notebook serves to select appropriate MAP words/phrases per MAP dimension and to analyze the outcome of the MAP survey.

For the selection of appropriate MAP words/phrases, various practitioner and academic articels are manually screened. The resulting list of potential words/phrases are then used to see which of them occur in the corpus of 10-k filings. Subsequently, 10-16 word clusters are formed per dimension which serve as the basis of the questionnair.

<div class="alert-warning">
Libraries
</div>

First, we Import all necessary libraries. These inlcude "os" and "pathlib" to set and handle working directories, "pandas", "numpy", "scipy", "statsmodels", and "itertools" for data handling and calculations, "pickle" to load and save the prepared data as memory efficient pickle files, "spacy" for NLP tasks, and "plotnine" to create plots/figures. 

In [1]:
#Load neccessary packages
import os
from pathlib import Path
import pickle
import pandas as pd
import numpy as np
import spacy
from gensim.models import KeyedVectors
import plotnine
from scipy.stats import entropy, ttest_ind, chisquare, chi2_contingency
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
from itertools import combinations

<div class="alert-warning">
Set the working directory
</div>

In [ ]:
os.chdir('../../../data')

<div class="alert-info">
Step 1: Assessing occurence of MAP words/phrases in 10-k filings
</div>

<div class="alert-warning">
Load the pre-processed corpus dataframe
</div>

In the first step of this analysis, we will clean the filings by dropping empty entries and joining the filing text into one string.

In [ ]:
#Load corpus dataframe

with open('W2V/Intermediate_datasets/Corpus_df_W2V_v4.pkl','rb') as path_name:
    corpus_df = pickle.load(path_name)

corpus_df = corpus_df.drop(columns=['filing_text'])

# Define a function to clean and join the text
def clean_and_join(text_list):
    cleaned_text = [text for text in text_list if text is not None]
    return ' '.join(cleaned_text)

# Apply the function to the "NER_filing_text" column
corpus_df['NER_filing_text'] = corpus_df['NER_filing_text'].apply(clean_and_join)

# Display the updated dataframe
corpus_df[['NER_filing_text']].head()

,NER_filing_text
0,[NER_ORG] abbott_laboratories is an [NER_GPE]_...
1,business cautionary_statement regarding_forwar...
2,general business_description of [NER_ORG] we o...
3,[NER_ORG]_formed_in_[NER_DATE] [NER_ORG] is a ...
4,[NER_ORG] the registrant is a [NER_GPE]_corpor...


Next, we can load the list of potential MAP words/phrases and count the occurence of these words in the 10-k filings.

In [ ]:
#Load the MAP tokens in a separat list-type object
map_seed_word_dictionary = pd.read_csv("W2V/Dictionary_creation/MAP_Seed_Words_Dictionary_final.csv", sep=";")

MAP_tokens = list(map_seed_word_dictionary["Seed_word_final"])

#Load the standard english web-trained large ("en_core_web_lg") spacy model/pipeline (disable "tagger", "parser", "ner", "textcat", and "lemmatizer")
nlp = spacy.load("en_core_web_lg", disable=["tagger", "parser", "ner", "textcat", "lemmatizer"])
#Since we just use the tokenizer we can increase the maximum length of a document withouth running into memory issues
nlp.max_length = 3000000

#Initialize PhraseMatcher and add the MAP phrase patterns / tokens.
matcher = spacy.matcher.PhraseMatcher(nlp.vocab)
patterns = [nlp.make_doc(token) for token in MAP_tokens]
matcher.add("PHRASES", patterns)

#Define a function to count the MAP tokens in a given text
def update_word_counts(doc, word_dict):
    matches = matcher(doc)
    for match_id, start, end in matches:
        span = doc[start:end].text
        if span in word_dict:
            word_dict[span] += 1

#Create a new column in which the dictionary with the respective frequencies will be saved in
corpus_df["MAP_token_count"] = None

#Run a loop to count the MAP-related (compound) tokens frequency
for doc in range(0,len(corpus_df["NER_filing_text"])):

    if doc%50 == 0:
        print(f"{doc} documents have been processed")

    #Set counter to 0 for all tokens
    word_dict = {token: 0 for token in MAP_tokens}
    #Load the filing document into the spacy NLP pipeline
    doc_nlp = nlp(corpus_df.loc[doc,"NER_filing_text"])
    #Update the word_dict with the word frequency of the considered document
    update_word_counts(doc_nlp, word_dict)
    #Save the dictionary with the word frequencies to the new column "MAP_token_count" in corpus_df
    corpus_df.at[doc,"MAP_token_count"] = word_dict

#save the updated corpus_df as a pickle file
corpus_df.to_pickle("W2V/Intermediate_datasets/Corpus_df_W2V_v4.pkl")

#Create the term matrix as a separate dataframe 
term_matrix = pd.DataFrame(dict(corpus_df["MAP_token_count"])).transpose()

#Save the term matrix as an excel file
term_matrix.to_excel("W2V/Dictionary_creation/term_matrix_token_level_final.xlsx")

#Save the summary statistic as an excel file
term_matrix_summary = term_matrix.describe()
term_matrix_summary.to_excel("W2V/Dictionary_creation/term_matrix_token_level_sum_final.xlsx")

del corpus_df, doc_nlp, word_dict, patterns, matcher, nlp, MAP_tokens

Next, we count the total occurences of the MAP words/phrases in the corpus, add the summary statistics, and finally add the top 30 synonyms according to the W2V model to the seed word dictionary. The resulting MAP seed word dictionary is exported to excel an manually examined to create the clusters for the questionnair.

In [ ]:
# Load w2v model from disk

w2v_file="W2V/Dictionary_creation/w2v_filing_text.model" 

w2v = KeyedVectors.load_word2vec_format(w2v_file, binary=True)

# Initialize the new columns with None or empty values
map_seed_word_dictionary["In_filing_text?"] = None
map_seed_word_dictionary["Synonyms"] = None
map_seed_word_dictionary["Count"] = 0

# Transpose the term matrix summary DataFrame
term_matrix_summary_transposed = term_matrix_summary.transpose()

# Merge the transposed term matrix summary with the map_seed_word_dictionary
map_seed_word_dictionary = pd.merge(map_seed_word_dictionary, term_matrix_summary_transposed, left_on='Seed_word_final', right_index=True, how='left')

# Delete "count" colmn from term_matrix_summary_transposed to avoid confusion
map_seed_word_dictionary = map_seed_word_dictionary.drop(columns=['count'])

# Count the occurrences of each seed word in the corpus and find synonyms
for i in range(0,len(map_seed_word_dictionary["Seed_word_final"])):
    seed_word = map_seed_word_dictionary["Seed_word_final"][i]
    map_seed_word_dictionary.at[i,"Count"] = sum(term_matrix[seed_word])
    try: 
        map_seed_word_dictionary.loc[i,"Synonyms"] = [w2v.most_similar(positive=seed_word, topn=30)]
        map_seed_word_dictionary.loc[i,"In_filing_text?"] = "yes"
    except KeyError:
        map_seed_word_dictionary.loc[i,"Synonyms"] = None
        map_seed_word_dictionary.loc[i,"In_filing_text?"] = "no"

#Save the updated map_seed_word_dictionary as an excel file
map_seed_word_dictionary.to_excel("W2V/Dictionary_creation/MAP_Seed_Words_Dictionary_analysis_final.xlsx", index=False)

#Free up memory
del w2v, term_matrix, term_matrix, term_matrix_summary_transposed, seed_word, i

<div class="alert-info">
Step 2: Analyze the MAP questionnair results
</div>

<div class="alert-warning">
Load the MAP questionair 
</div>

Once the questionnair is finished, we can load the results, analyze it, and pick the final seed words based on the term frequency.

In [ ]:
#Load MAP survey data 
MAP_questionnaire = pd.read_csv('Survey/MAP_survey_final.csv', delimiter=';')

#Filter to only approved responses (these are the prolific ones)
MAP_questionnaire = MAP_questionnaire[MAP_questionnaire["Status.1"] == "APPROVED"]

#If only Prolific participants with residency in US are to be considered
US_index = "No"

if US_index == "Yes":
    MAP_questionnaire = MAP_questionnaire[MAP_questionnaire["Country of residence"] == "United States"]


#Load variable mapping file
variable_mapping = pd.read_csv('Survey/MAP_survey_variables.csv', delimiter=';')

In [5]:
# Before taking a closer look at the data, we have a quick look at the summary statistics of the completion times of the survey. 

# Since the completion times are in seconds, we first convert them to minutes for better interpretability.
MAP_questionnaire["TIME_SUM"] = MAP_questionnaire["TIME_SUM"] / 60

# Display summary statistics for the completion times and Age
MAP_questionnaire[["TIME_SUM", "Age"]].describe()

,TIME_SUM,Age
count,70.000000,70.000000
mean,7.560952,39.357143
std,3.692786,10.065058
min,2.283333,22.000000
25%,4.787500,31.000000
50%,6.516667,38.000000
75%,9.533333,45.000000
max,16.950000,65.000000


<div class="alert-info">
Step 2.1: Describe Socio-demographic and Job-related data
</div>

In [ ]:
plotnine.options.figure_size = (12,7)

#1. Functional area
#Code mapping for functional areas
functional_area_mapping = [("GQ03_01", "Internal Reporting"), ("GQ03_02", "Financial Management and Analysis"), ("GQ03_03", "Cost Management and Control"), ("GQ03_04", "Corporate Investments"), ("GQ03_07", "Budgeting and Accounting"), ("GQ03_05", "Strategic Planning"), ("GQ03_06", "External Reporting (Financial or Non-Financial)")]

functional_area_counts = []

count_other = MAP_questionnaire["GQ03"].value_counts().get(-1, 0)    

functional_area_counts.append({'Functional Area': 'Other', 'Count': count_other})

for col, label in functional_area_mapping:
    functional_area_count = MAP_questionnaire[col].sum()
    functional_area_counts.append({'Functional Area': label, 'Count': functional_area_count})

ordered_labels = [label for col, label in functional_area_mapping] + ['Other']

functional_area_counts = pd.DataFrame(functional_area_counts, columns=['Functional Area', 'Count'])

functional_area_counts['Functional Area'] = pd.Categorical(functional_area_counts['Functional Area'], categories=ordered_labels, ordered=True)

plot = (plotnine.ggplot(functional_area_counts, plotnine.aes(x='Functional Area', y='Count')) +
        plotnine.geom_bar(stat='identity', fill='skyblue') +
        plotnine.theme(panel_background= plotnine.element_rect(fill='white'),
                           plot_background= plotnine.element_rect(fill='white'), 
                           axis_title_x=plotnine.element_text(size=18, weight='bold', vjust=-5),  
                           axis_title_y=plotnine.element_text(size=18, weight='bold'),
                           axis_text_x=plotnine.element_text(rotation=45, hjust=1, size=14, weight='bold'),  
                           axis_text_y=plotnine.element_text(size=14, weight='bold')) +
        plotnine.labs(y='Number of Respondents'))

# Save the plot and table in folder "Descriptives"
if US_index == "Yes":
    plot.save('Survey/Plots/Descriptives/functional_area_distribution_US.png', width=19.2, height=9.67, dpi=300)
    functional_area_counts.to_excel('Survey/Tables/Descriptives/functional_area_distribution_US.xlsx', index=False)
else:
    plot.save('Survey/Plots/Descriptives/functional_area_distribution.png', width=19.2, height=9.67, dpi=300)
    functional_area_counts.to_excel('Survey/Tables/Descriptives/functional_area_distribution.xlsx', index=False)

del functional_area_counts, functional_area_mapping

#2. Job role
job_role_mapping = {1: "CEO, CFO, CIO, COO", 2: "Vice President of Finance or Director of Finance", 3: "Manager (Accounting or Finance)", 4: "Controller", -1: "Other"}

ordered_labels = [job_role_mapping[i] for i in [1, 2, 3, 4, -1]]
# Create bar plot for job role distribution
job_role_counts = MAP_questionnaire['GQ04_2'].value_counts().reset_index()
job_role_counts.columns = ['Job Role', 'Count']
job_role_counts['Job Role'] = job_role_counts['Job Role'].map(job_role_mapping)
job_role_counts['Job Role'] = pd.Categorical(job_role_counts['Job Role'], categories=ordered_labels, ordered=True)

plot = (plotnine.ggplot(job_role_counts, plotnine.aes(x='Job Role', y='Count')) +
        plotnine.geom_bar(stat='identity', fill='skyblue') +
        plotnine.theme(panel_background= plotnine.element_rect(fill='white'),
                           plot_background= plotnine.element_rect(fill='white'), 
                           axis_title_x=plotnine.element_text(size=18, weight='bold', vjust=-5),  
                           axis_title_y=plotnine.element_text(size=18, weight='bold'),
                           axis_text_x=plotnine.element_text(rotation=45, hjust=1, size=14, weight='bold'),  
                           axis_text_y=plotnine.element_text(size=14, weight='bold')) +
        plotnine.labs(y='Number of Respondents'))

# Save the plot and table
if US_index == "Yes":
    plot.save('Survey/Plots/Descriptives/job_role_distribution_US.png', width=19.2, height=9.67, dpi=300)
    job_role_counts.to_excel('Survey/Tables/Descriptives/job_role_distribution_US.xlsx', index=False)
else:
    plot.save('Survey/Plots/Descriptives/job_role_distribution.png', width=19.2, height=9.67, dpi=300)
    job_role_counts.to_excel('Survey/Tables/Descriptives/job_role_distribution.xlsx', index=False)

del job_role_counts, job_role_mapping

#3. Time in current position
position_time_mapping = {1: "Less than 1 year", 2: "1-3 years", 3: "4-6 years", 4: "7-10 years", 5: "More than 10 years"}

ordered_labels = [position_time_mapping[i] for i in [1, 2, 3, 4, 5]]

# Create bar plot for time in current position distribution
position_time_counts = MAP_questionnaire['GQ05'].value_counts().reset_index()
position_time_counts.columns = ['Time in Current Position', 'Count']
position_time_counts['Time in Current Position'] = position_time_counts['Time in Current Position'].map(position_time_mapping)
position_time_counts['Time in Current Position'] = pd.Categorical(position_time_counts['Time in Current Position'], categories=ordered_labels, ordered=True)

plot = (plotnine.ggplot(position_time_counts, plotnine.aes(x='Time in Current Position', y='Count')) +
        plotnine.geom_bar(stat='identity', fill='skyblue') +
        plotnine.theme(panel_background= plotnine.element_rect(fill='white'),
                           plot_background= plotnine.element_rect(fill='white'), 
                           axis_title_x=plotnine.element_text(size=18, weight='bold', vjust=-5),  
                           axis_title_y=plotnine.element_text(size=18, weight='bold'),
                           axis_text_x=plotnine.element_text(rotation=45, hjust=1, size=14, weight='bold'),  
                           axis_text_y=plotnine.element_text(size=14, weight='bold')) +
        plotnine.labs(y='Number of Respondents'))

# Save the plot and table
if US_index == "Yes":
    plot.save('Survey/Plots/Descriptives/position_time_distribution_US.png', width=19.2, height=9.67, dpi=300)
    position_time_counts.to_excel('Survey/Tables/Descriptives/position_time_distribution_US.xlsx', index=False)
else:
    plot.save('Survey/Plots/Descriptives/position_time_distribution.png', width=19.2, height=9.67, dpi=300)
    position_time_counts.to_excel('Survey/Tables/Descriptives/position_time_distribution.xlsx', index=False)
del position_time_counts, position_time_mapping

#4. Time in MA field
field_time_mapping = {1: "Less than 1 year", 2: "1-3 years", 3: "4-6 years", 4: "7-10 years", 5: "More than 10 years", -1: "Not in MA field"}

ordered_labels = [field_time_mapping[i] for i in [1, 2, 3, 4, 5, -1]]

# Create bar plot for time in MA field distribution
field_time_counts = MAP_questionnaire['GQ06_2'].value_counts().reset_index()
field_time_counts.columns = ['Time in MA Field', 'Count']
field_time_counts['Time in MA Field'] = field_time_counts['Time in MA Field'].map(field_time_mapping)
field_time_counts['Time in MA Field'] = pd.Categorical(field_time_counts['Time in MA Field'], categories=ordered_labels, ordered=True)

plot = (plotnine.ggplot(field_time_counts, plotnine.aes(x='Time in MA Field', y='Count')) +
        plotnine.geom_bar(stat='identity', fill='skyblue') +
        plotnine.theme(panel_background= plotnine.element_rect(fill='white'),
                           plot_background= plotnine.element_rect(fill='white'), 
                           axis_title_x=plotnine.element_text(size=18, weight='bold', vjust=-5),  
                           axis_title_y=plotnine.element_text(size=18, weight='bold'),
                           axis_text_x=plotnine.element_text(rotation=45, hjust=1, size=14, weight='bold'),  
                           axis_text_y=plotnine.element_text(size=14, weight='bold')) +
        plotnine.labs(y='Number of Respondents'))

# Save the plot and table
if US_index == "Yes":
    plot.save('Survey/Plots/Descriptives/field_time_distribution_US.png', width=19.2, height=9.67, dpi=300)
    field_time_counts.to_excel('Survey/Tables/Descriptives/field_time_distribution_US.xlsx', index=False)
else:
    plot.save('Survey/Plots/Descriptives/field_time_distribution.png', width=19.2, height=9.67, dpi=300)
    field_time_counts.to_excel('Survey/Tables/Descriptives/field_time_distribution.xlsx', index=False)
del field_time_counts, field_time_mapping

#5. Company size 

# Change variable to string and order the categories
MAP_questionnaire['Company size'] = MAP_questionnaire['Company size'].astype(str)
ordered_labels = ['1-9', '10-49', '50-249', '250-999', '1000+']
#Create bar plot for company size distribution
company_size_counts = MAP_questionnaire['Company size'].value_counts().reset_index()
company_size_counts.columns = ['Company Size', 'Count']
company_size_counts['Company Size'] = pd.Categorical(company_size_counts['Company Size'], categories=ordered_labels, ordered=True)
plot = (plotnine.ggplot(company_size_counts, plotnine.aes(x='Company Size', y='Count')) +
        plotnine.geom_bar(stat='identity', fill='skyblue') +
        plotnine.theme(panel_background= plotnine.element_rect(fill='white'),
                           plot_background= plotnine.element_rect(fill='white'), 
                           axis_title_x=plotnine.element_text(size=18, weight='bold', vjust=-5),  
                           axis_title_y=plotnine.element_text(size=18, weight='bold'),                           
                           axis_text_x=plotnine.element_text(rotation=45, hjust=1, size=14, weight='bold'),  
                           axis_text_y=plotnine.element_text(size=14, weight='bold')) +
        plotnine.labs(y='Number of Respondents'))

# Save the plot and table
if US_index == "Yes":
    plot.save('Survey/Plots/Descriptives/company_size_distribution_US.png', width=19.2, height=9.67, dpi=300)
    company_size_counts.to_excel('Survey/Tables/Descriptives/company_size_distribution_US.xlsx', index=False)
else:
    plot.save('Survey/Plots/Descriptives/company_size_distribution.png', width=19.2, height=9.67, dpi=300)
    company_size_counts.to_excel('Survey/Tables/Descriptives/company_size_distribution.xlsx', index=False)
del company_size_counts

#6. Company type

# Rename company type categories
company_type_mapping = {"Micro enterprise": "Micro enterprise", "Small and medium-sized enterprises (SME)": "SME", "Large private enterprise": "Large private enterprise", "Publicly listed/traded enterprise (e.g. listed on a stock exchange)": "Publicly listed", "Other": "Other"}

ordered_labels = [company_type_mapping[key] for key in ["Micro enterprise", "Small and medium-sized enterprises (SME)", "Large private enterprise", "Publicly listed/traded enterprise (e.g. listed on a stock exchange)", "Other"]]

#Create bar plot for company type distribution
company_type_counts = MAP_questionnaire['Company type'].value_counts().reset_index()
company_type_counts.columns = ['Company Type', 'Count']
company_type_counts['Company Type'] = company_type_counts['Company Type'].map(company_type_mapping)
company_type_counts['Company Type'] = pd.Categorical(company_type_counts['Company Type'], categories=ordered_labels, ordered=True)

plot = (plotnine.ggplot(company_type_counts, plotnine.aes(x='Company Type', y='Count')) +
        plotnine.geom_bar(stat='identity', fill='skyblue') +
        plotnine.theme(panel_background= plotnine.element_rect(fill='white'),
                           plot_background= plotnine.element_rect(fill='white'), 
                            axis_title_x=plotnine.element_text(size=18, weight='bold', vjust=-5),
                            axis_title_y=plotnine.element_text(size=18, weight='bold'),
                                axis_text_x=plotnine.element_text(rotation=45, hjust=1, size=14, weight='bold'),  
                                axis_text_y=plotnine.element_text(size=14, weight='bold')) +
        plotnine.labs(y='Number of Respondents'))       

# Save the plot and table
if US_index == "Yes":
    plot.save('Survey/Plots/Descriptives/company_type_distribution_US.png', width=19.2, height=9.67, dpi=300)
    company_type_counts.to_excel('Survey/Tables/Descriptives/company_type_distribution_US.xlsx', index=False)
else:
    plot.save('Survey/Plots/Descriptives/company_type_distribution.png', width=19.2, height=9.67, dpi=300)
    company_type_counts.to_excel('Survey/Tables/Descriptives/company_type_distribution.xlsx', index=False)
del company_type_counts

#7. Industry sector

#Create bar plot for industry sector distribution
industry_sector_counts = MAP_questionnaire['Industry'].value_counts().reset_index()
industry_sector_counts.columns = ['Industry Sector', 'Count']

plot = (plotnine.ggplot(industry_sector_counts, plotnine.aes(x='Industry Sector', y='Count')) +
        plotnine.geom_bar(stat='identity', fill='skyblue') +
        plotnine.theme(panel_background= plotnine.element_rect(fill='white'),
                           plot_background= plotnine.element_rect(fill='white'), 
                            axis_title_x=plotnine.element_text(size=18, weight='bold', vjust=-5),
                            axis_title_y=plotnine.element_text(size=18, weight='bold'),
                                axis_text_x=plotnine.element_text(rotation=45, hjust=1, size=14, weight='bold'),  
                                axis_text_y=plotnine.element_text(size=14, weight='bold')) +
        plotnine.labs(y='Number of Respondents'))

# Save the plot and table
if US_index == "Yes":
    plot.save('Survey/Plots/Descriptives/industry_sector_distribution_US.png', width=19.2, height=9.67, dpi=300)
    industry_sector_counts.to_excel('Survey/Tables/Descriptives/industry_sector_distribution_US.xlsx', index=False)
else:
    plot.save('Survey/Plots/Descriptives/industry_sector_distribution.png', width=19.2, height=9.67, dpi=300)
    industry_sector_counts.to_excel('Survey/Tables/Descriptives/industry_sector_distribution.xlsx', index=False)
del industry_sector_counts

#8. Employment Status
employment_status_counts = MAP_questionnaire['Employment status'].value_counts().reset_index()
employment_status_counts.columns = ['Employment Status', 'Count']

plot = (plotnine.ggplot(employment_status_counts, plotnine.aes(x='Employment Status', y='Count')) +
        plotnine.geom_bar(stat='identity', fill='skyblue') +
        plotnine.theme(panel_background= plotnine.element_rect(fill='white'),
                           plot_background= plotnine.element_rect(fill='white'), 
                            axis_title_x=plotnine.element_text(size=18, weight='bold', vjust=-5),
                            axis_title_y=plotnine.element_text(size=18, weight='bold'),
                                axis_text_x=plotnine.element_text(rotation=45, hjust=1, size=14, weight='bold'),  
                                axis_text_y=plotnine.element_text(size=14, weight='bold')) +
        plotnine.labs(y='Number of Respondents'))

# Save the plot and table
if US_index == "Yes":
    plot.save('Survey/Plots/Descriptives/employment_status_distribution_US.png', width=19.2, height=9.67, dpi=300)
    employment_status_counts.to_excel('Survey/Tables/Descriptives/employment_status_distribution_US.xlsx', index=False)
else:
    plot.save('Survey/Plots/Descriptives/employment_status_distribution.png', width=19.2, height=9.67, dpi=300)
    employment_status_counts.to_excel('Survey/Tables/Descriptives/employment_status_distribution.xlsx', index=False)
del employment_status_counts

#9. Respondent Age

#create bar plot for age distribution with age groups of 5 years
age_bins = [18, 24, 29, 34, 39, 44, 49, 54, 59, 64, 69]
age_labels = ['18-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69']
MAP_questionnaire['Age Group'] = pd.cut(np.asarray(MAP_questionnaire['Age'], dtype=int), bins=age_bins, labels=age_labels, right=True)
age_group_counts = MAP_questionnaire['Age Group'].value_counts().reset_index()
age_group_counts.columns = ['Age Group', 'Count']
age_group_counts = age_group_counts.sort_values(by='Age Group')

plot = (plotnine.ggplot(age_group_counts, plotnine.aes(x='Age Group', y='Count')) +
        plotnine.geom_bar(stat='identity', fill='skyblue') +
        plotnine.theme(panel_background= plotnine.element_rect(fill='white'),
                           plot_background= plotnine.element_rect(fill='white'), 
                            axis_title_x=plotnine.element_text(size=18, weight='bold', vjust=-5),
                            axis_title_y=plotnine.element_text(size=18, weight='bold'),
                                axis_text_x=plotnine.element_text(rotation=45, hjust=1, size=14, weight='bold'),  
                                axis_text_y=plotnine.element_text(size=14, weight='bold')) +
        plotnine.labs(y='Number of Respondents'))
# Save the plot and table
if US_index == "Yes":
    plot.save('Survey/Plots/Descriptives/age_distribution_US.png', width=19.2, height=9.67, dpi=300)
    age_group_counts.to_excel('Survey/Tables/Descriptives/age_distribution_US.xlsx', index=False)
else:
    plot.save('Survey/Plots/Descriptives/age_distribution.png', width=19.2, height=9.67, dpi=300)
    age_group_counts.to_excel('Survey/Tables/Descriptives/age_distribution.xlsx', index=False)
del age_group_counts

#10. Respondent Gender
gender_counts = MAP_questionnaire['Sex'].value_counts().reset_index()
gender_counts.columns = ['Gender', 'Count']
plot = (plotnine.ggplot(gender_counts, plotnine.aes(x='Gender', y='Count')) +
        plotnine.geom_bar(stat='identity', fill='skyblue') +
        plotnine.theme(panel_background= plotnine.element_rect(fill='white'),
                           plot_background= plotnine.element_rect(fill='white'), 
                            axis_title_x=plotnine.element_text(size=18, weight='bold', vjust=-5),
                            axis_title_y=plotnine.element_text(size=18, weight='bold'),
                                axis_text_x=plotnine.element_text(rotation=45, hjust=1, size=14, weight='bold'),  
                                axis_text_y=plotnine.element_text(size=14, weight='bold')) +
        plotnine.labs(y='Number of Respondents'))

# Save the plot and table
if US_index == "Yes":
    plot.save('Survey/Plots/Descriptives/gender_distribution_US.png', width=19.2, height=9.67, dpi=300)
    gender_counts.to_excel('Survey/Tables/Descriptives/gender_distribution_US.xlsx', index=False)
else:
    plot.save('Survey/Plots/Descriptives/gender_distribution.png', width=19.2, height=9.67, dpi=300)
    gender_counts.to_excel('Survey/Tables/Descriptives/gender_distribution.xlsx', index=False)
del gender_counts

#11. Country of Residence
country_counts = MAP_questionnaire['Country of residence'].value_counts().reset_index()
country_counts.columns = ['Country of Residence', 'Count']
plot = (plotnine.ggplot(country_counts, plotnine.aes(x='Country of Residence', y='Count')) +
        plotnine.geom_bar(stat='identity', fill='skyblue') +
        plotnine.theme(panel_background= plotnine.element_rect(fill='white'),
                           plot_background= plotnine.element_rect(fill='white'), 
                            axis_title_x=plotnine.element_text(size=18, weight='bold', vjust=-5),
                            axis_title_y=plotnine.element_text(size=18, weight='bold'),
                                axis_text_x=plotnine.element_text(rotation=45, hjust=1, size=14, weight='bold'),  
                                axis_text_y=plotnine.element_text(size=14, weight='bold')) +
        plotnine.labs(y='Number of Respondents'))

# Save the plot and table
if US_index == "Yes":
    plot.save('Survey/Plots/Descriptives/country_of_residence_distribution_US.png', width=19.2, height=9.67, dpi=300)
    country_counts.to_excel('Survey/Tables/Descriptives/country_of_residence_distribution_US.xlsx', index=False)
else:
    plot.save('Survey/Plots/Descriptives/country_of_residence_distribution.png', width=19.2, height=9.67, dpi=300)
    country_counts.to_excel('Survey/Tables/Descriptives/country_of_residence_distribution.xlsx', index=False)

del country_counts, plot

<div class="alert-info">
Step 2.2: Analyze Respodents MAP item choices
</div>

The Questionnair entails forced-choice questions, where respondents should select 5 words/phrases that they believe are most relevant for reflecting whether a firm applies MAPs in the corresponding area. To analyze these questions we have a look at the selection frequency, the selection concentration, and the respondents' agreement of the selected items.

First, we load the survey data and select appropriate filter options.

In [ ]:
#Load MAP survey data 
MAP_questionnaire = pd.read_csv('Survey/MAP_survey_final.csv', delimiter=';')

#Filter to only approved responses (these are the prolific ones)
MAP_questionnaire = MAP_questionnaire[MAP_questionnaire["Status.1"] == "APPROVED"]

#Optional filters
# Filter to only Prolific participants with residency in US are to be considered
US_index = "Yes"
#Filter to only Prolific participants with more than 3 years of experience in MA field
Experienced_index = "No"
#Filter to only controllers (managerial roles are typically generalists in MA)
controller_index = "No"


#If only Prolific participants with residency in US, at least 4 years of experience in the MA field, or controller roles are to be considered
if US_index == "Yes":
    MAP_questionnaire = MAP_questionnaire[MAP_questionnaire["Country of residence"] == "United States"]
if Experienced_index == "Yes":
    MAP_questionnaire = MAP_questionnaire[MAP_questionnaire["GQ06_2"] > 2] # Optional: Filter to only respondents with more than 3 years of experience in MA field
if controller_index == "Yes":
    MAP_questionnaire = MAP_questionnaire[MAP_questionnaire["GQ04_2"] == 4] # Optional: Filter to only controllers (managerial roles are typically generalists in MA)

#Always remove respondents who are not in the MA field
MAP_questionnaire = MAP_questionnaire[MAP_questionnaire["GQ06_2"] != -1]

#Load variable mapping file
variable_mapping = pd.read_csv('Survey/MAP_survey_variables.csv', delimiter=';')

Now, we run the analyses per MAP dimension.

In [ ]:
#Mapping of MAP dimensions to question/item IDs
dimension_mapping = {"Budgeting/Planning": ["MP01_01","MP01_02","MP01_03", "MP01_04", "MP01_05", "MP01_06", "MP01_07", "MP01_08", "MP01_09", "MP01_10"],
                     "Cost": ["MP02_01","MP02_02","MP02_03", "MP02_04", "MP02_05", "MP02_06", "MP02_07", "MP02_08", "MP02_09", "MP02_10", "MP02_11"],
                     "Financing/Investment": ["MP05_01","MP05_02","MP05_03", "MP05_04", "MP05_05", "MP05_06", "MP05_07", "MP05_08", "MP05_09", "MP05_10", "MP05_11", "MP05_12"],
                     "Operations": ["MP06_01","MP06_02","MP06_03", "MP06_04", "MP06_05", "MP06_06", "MP06_07", "MP06_08", "MP06_09", "MP06_10", "MP06_11", "MP06_12", "MP06_13", "MP06_14", "MP06_15", "MP06_16"],
                     "Performance/Internal Reporting": ["MP07_01","MP07_02","MP07_03", "MP07_04", "MP07_05", "MP07_06", "MP07_07", "MP07_08", "MP07_09", "MP07_11", "MP07_12", "MP07_13", "MP07_14"],
                     "Pricing/Revenue Management": ["MP08_01","MP08_02","MP08_03", "MP08_04", "MP08_05", "MP08_06", "MP08_07", "MP08_08", "MP08_09", "MP08_10", "MP08_11"],
                     "Risk/Internal Control": ["MP09_01","MP09_02","MP09_03", "MP09_04", "MP09_05", "MP09_06", "MP09_07", "MP09_08", "MP09_09", "MP09_10", "MP09_11", "MP09_12", "MP09_13", "MP09_14", "MP09_15"],
                     "Strategy": ["MP10_01","MP10_02", "MP10_05", "MP10_06", "MP10_07", "MP10_08", "MP10_09", "MP10_10", "MP10_11", "MP10_12", "MP10_13", "MP10_14", "MP10_16", "MP10_17", "MP10_18", "MP10_19"]}

#Initialize lists to store results
response_summary = []
dimension_summary = []

#Iterate over each dimension and perform analysis
for dimension, questions in dimension_mapping.items():
    #Filter the survey data for the relevant questions
    MAP_questions_subset = MAP_questionnaire[questions]

    #Number of respondents
    n_respondents = MAP_questions_subset.shape[0]

    #Calculate the proportion of respondents selecting each answer option per question (selection frequency)
    #It reflects which aspects within each dimension are considered most important by the respondents
    response_distribution = MAP_questions_subset.mean().reset_index().sort_values(by=0, ascending=False)
    response_distribution.columns = ['Response', 'Proportion']

    #Binary variance for each question (p * (1 - p)) to assess the dispersion of responses
    # A higher variance indicates a more diverse set of responses, while a lower variance suggests consensus among respondents.
    # max variance is 0.25 when p = 0.5
    response_distribution['Binary_Variance'] = response_distribution['Proportion'] * (1 - response_distribution['Proportion'])

    #Add the standard deviation for additional context
    response_distribution['Std_Dev'] = MAP_questions_subset.std().reset_index()[0]

    #Calculate entropy for each Response (uncertainty in selection)
    response_distribution["prob_0"] = 1 - response_distribution['Proportion']

    # Entropy measures the degree of uncertainty or randomness in the selection probabilities.
    # Lower entropy indicates more concentrated selections, while higher entropy indicates more dispersed selections.
    response_distribution['Entropy'] = response_distribution.apply(lambda row: entropy([row['prob_0'], row['Proportion']], base=2), axis=1)
    response_distribution = response_distribution.drop(columns=['prob_0'])

    # Calculate the consensus index for each Response (1 - Entropy)
    # The consensus index is a measure of agreement among respondents regarding the importance of each aspect within a dimension. 
    # It is calculated as 1 minus the entropy, where a higher consensus index indicates greater agreement among respondents, while a lower consensus index suggests more diverse opinions.
    response_distribution['Consensus_Index'] = 1 - response_distribution['Entropy']

    # Replace the response codes with the actual question text from variable_mapping
    response_distribution = response_distribution.merge(variable_mapping[['VAR', 'LABEL']], left_on='Response', right_on='VAR', how='left').drop(columns=['VAR'])
    # Adjust label column by splitting at the first ":" symbole and keeping the second part
    response_distribution['LABEL'] = response_distribution['LABEL'].apply(lambda x: x.split(":", 1)[1].strip() if pd.notnull(x) else x)

    # Print response summary for each dimension
    print(f"Dimension: {dimension}")
    print(response_distribution)

    #Create a bar plot for the response distribution (with number of respondents indicated in the title)
    plot = (plotnine.ggplot(response_distribution, plotnine.aes(x='reorder(LABEL, -Proportion)', y='Proportion')) +
            plotnine.geom_bar(stat='identity', fill='skyblue') +
            plotnine.coord_flip() +
            plotnine.labs(x='Response', y='Proportion', title=f'Number of Respondents: {n_respondents}') +
            plotnine.theme(panel_background= plotnine.element_rect(fill='white'),
                           plot_background= plotnine.element_rect(fill='white')))
    
    if US_index == "Yes" and Experienced_index == "No" and controller_index == "No":
        plot.save(f'Survey/Plots/Response/{dimension.replace("/", "_")}_Response_Distribution_US.png', width=10, height=6, dpi=300)
    if US_index == "Yes" and Experienced_index == "Yes" and controller_index == "No":
        plot.save(f'Survey/Plots/Response/{dimension.replace("/", "_")}_Response_Distribution_US_experienced.png', width=10, height=6, dpi=300)
    if US_index == "Yes" and Experienced_index == "No" and controller_index == "Yes":
        plot.save(f'Survey/Plots/Response/{dimension.replace("/", "_")}_Response_Distribution_US_controller.png', width=10, height=6, dpi=300)
    if US_index == "Yes" and Experienced_index == "Yes" and controller_index == "Yes":
        plot.save(f'Survey/Plots/Response/{dimension.replace("/", "_")}_Response_Distribution_US_experienced_controller.png', width=10, height=6, dpi=300)
    if US_index == "No" and Experienced_index == "No" and controller_index == "Yes":
        plot.save(f'Survey/Plots/Response/{dimension.replace("/", "_")}_Response_Distribution_controller.png', width=10, height=6, dpi=300)
    if US_index == "No" and Experienced_index == "Yes" and controller_index == "Yes":
        plot.save(f'Survey/Plots/Response/{dimension.replace("/", "_")}_Response_Distribution_experienced_controller.png', width=10, height=6, dpi=300)
    if US_index == "No" and Experienced_index == "Yes" and controller_index == "No":
        plot.save(f'Survey/Plots/Response/{dimension.replace("/", "_")}_Response_Distribution_experienced.png', width=10, height=6, dpi=300)
    if US_index == "No" and Experienced_index == "No" and controller_index == "No":
        plot.save(f'Survey/Plots/Response/{dimension.replace("/", "_")}_Response_Distribution.png', width=10, height=6, dpi=300)
    else:
        print("No matching condition for saving the plot.")
    del plot
    #Append results to response summary list
    response_summary.append((dimension, response_distribution))

    #Now calculate additional metrics for each dimension
    ##############################################
    # Observed Entropy (selection concentration).# 
    ##############################################
    # Entropy measures the degree of dispersion in the selection probabilities.
    # Lower entropy indicates more concentrated selections, while higher entropy indicates more dispersed selections.
    # Maximum entropy occurs when all items are equally likely to be selected.
    # Minimum entropy cannot be zero in forced-choice settings, as respondents must select a fixed number of items.
    # Minimum entropy occurs when all respondents select the same set of items.

    #1. Selection probabilities
    p = MAP_questions_subset.mean() 
    p_norm = p / p.sum()  #normalize to sum to 1

    #2. Observed Entropy (H_obs), Maximum Entropy (H_max), Minimum Entropy (H_min)
    H_obs = entropy(p_norm, base=2)
    n_terms = len(p) #number of items in the dimension
    H_max = np.log2(n_terms) #Maximum entropy when all items are equally likely to be selected
    p_min = np.zeros(n_terms) 
    p_min[:5] = 1/5  #Assuming k=5 selections per respondent
    H_min = entropy(p_min, base=2)

    #3. Normalized Entropy (H_obs_norm) and Relative Entropy (H_obs_rel)
    # Scales the observed entropy between the minimum and maximum possible entropy for the given forced-choice setup.
    # where 0 indicates the most concentrated selection pattern (minimum entropy)
    # and 1 indicates the most dispersed selection pattern (maximum entropy).
    H_obs_norm = (H_obs - H_min) / (H_max - H_min)

    print("Observed Entropy:", H_obs)
    print("Maximum Entropy:", H_max)
    print("Normalized Observed Entropy:", H_obs_norm)

    ##########################################
    # Chi-Square Test (test for randomness). # 
    ##########################################
    # The chi-square test assesses whether the observed selection distribution significantly differs from a random distribution.
    # A significant result suggests that respondents are not selecting items randomly, indicating a more concentrated selection pattern.
    observed_counts = MAP_questions_subset.sum().values
    expected_counts = np.full(n_terms, observed_counts.sum() / n_terms) 
    chi2_stat, p_value = chisquare(f_obs=observed_counts, f_exp=expected_counts)
    dof = n_terms - 1  # degrees of freedom

    print("Chi-Square Statistic:", chi2_stat)
    print("P-value:", p_value)
    print("Degrees of Freedom:", dof)

    ###########################################
    # Fleiss' Kappa (inter-rater reliability).#
    ###########################################
    # Fleiss' Kappa measures the agreement among multiple respondents in their selections.
    # Higher Fleiss' Kappa values indicate greater agreement beyond what would be expected by chance
    aggregate = aggregate_raters(MAP_questions_subset)
    f_kappa = fleiss_kappa(aggregate_raters(MAP_questions_subset.T)[0], method='fleiss')

    print("Fleiss' Kappa:", f_kappa)    

    #Append results to dimension summary list
    dimension_summary.append({
        'Dimension': dimension,
        'Observed_Entropy': H_obs,
        'Maximum_Entropy': H_max,
        'Normalized_Entropy': H_obs_norm,
        'Chi_Square_Statistic': chi2_stat,
        'Chi_Square_P_value': p_value,
        'Chi_Square_Degrees_of_Freedom': dof,
        'Fleiss_kappa': f_kappa
    })


#Combine all response and dimension summaries into a DataFrame and save to excel
all_responses_df = pd.concat([df.assign(Dimension=dim) for dim, df in response_summary], ignore_index=True)
dimension_summary_df = pd.DataFrame(dimension_summary)
if US_index == "Yes" and Experienced_index == "No" and controller_index == "No":
    all_responses_df.to_excel('Survey/Tables/Response/MAP_Response_Summary_US.xlsx', index=False)
    dimension_summary_df.to_excel('Survey/Tables/Dimension/MAP_Dimension_Summary_US.xlsx', index=False)
if US_index == "Yes" and Experienced_index == "Yes" and controller_index == "No":
    all_responses_df.to_excel('Survey/Tables/Response/MAP_Response_Summary_US_experienced.xlsx', index=False)
    dimension_summary_df.to_excel('Survey/Tables/Dimension/MAP_Dimension_Summary_US_experienced.xlsx', index=False)
if US_index == "Yes" and Experienced_index == "No" and controller_index == "Yes":
    all_responses_df.to_excel('Survey/Tables/Response/MAP_Response_Summary_US_controller.xlsx', index=False)
    dimension_summary_df.to_excel('Survey/Tables/Dimension/MAP_Dimension_Summary_US_controller.xlsx', index=False)
if US_index == "Yes" and Experienced_index == "Yes" and controller_index == "Yes":
    all_responses_df.to_excel('Survey/Tables/Response/MAP_Response_Summary_US_experienced_controller.xlsx', index=False)
    dimension_summary_df.to_excel('Survey/Tables/Dimension/MAP_Dimension_Summary_US_experienced_controller.xlsx', index=False)
if US_index == "No" and Experienced_index == "No" and controller_index == "Yes":
    all_responses_df.to_excel('Survey/Tables/Response/MAP_Response_Summary_controller.xlsx', index=False)
    dimension_summary_df.to_excel('Survey/Tables/Dimension/MAP_Dimension_Summary_controller.xlsx', index=False)
if US_index == "No" and Experienced_index == "Yes" and controller_index == "Yes":
    all_responses_df.to_excel('Survey/Tables/Response/MAP_Response_Summary_experienced_controller.xlsx', index=False)
    dimension_summary_df.to_excel('Survey/Tables/Dimension/MAP_Dimension_Summary_experienced_controller.xlsx', index=False)
if US_index == "No" and Experienced_index == "Yes" and controller_index == "No":
    all_responses_df.to_excel('Survey/Tables/Response/MAP_Response_Summary_experienced.xlsx', index=False)
    dimension_summary_df.to_excel('Survey/Tables/Dimension/MAP_Dimension_Summary_experienced.xlsx', index=False)
if US_index == "No" and Experienced_index == "No" and controller_index == "No":
    all_responses_df.to_excel('Survey/Tables/Response/MAP_Response_Summary.xlsx', index=False)
    dimension_summary_df.to_excel('Survey/Tables/Dimension/MAP_Dimension_Summary.xlsx', index=False)
else:
    print("No matching condition for saving the summary files.")

#del MAP_questions_subset, response_distribution, dimension, questions, p, H_obs, H_max, n_permutations, rng, H_null, p_rand, mean_jaccard, response_summary, dimension_summary, all_responses_df, dimension_summary_df

Last, we have a look at potential differences in the selection frequencies across countries (US vs Non-US), job positions (Controller vs. Non-Controller), and MA experiences (more than 3 years vs. less).

In [ ]:
#Load MAP survey data 
MAP_questionnaire = pd.read_csv('Survey/MAP_survey_final.csv', delimiter=';')

#Filter to only approved responses (these are the prolific ones)
MAP_questionnaire = MAP_questionnaire[MAP_questionnaire["Status.1"] == "APPROVED"]

# Remove observations where the respondents does not work in MA field
MAP_questionnaire = MAP_questionnaire[MAP_questionnaire["GQ06_2"] != -1]

1. Differences in selection frequencies across countries (US vs. Non-US)

In [49]:
# create one table that contains the dimension comparison, which can be extended to other demographic variables as well (e.g. experience, role)
dimension_comparison = []

country_col = "Country of residence"
country_means_t = None
#Iterate over each dimension and perform analysis of differences by country at the item and dimension level (t-tests and chi-square tests)
for dimension, questions in dimension_mapping.items():    
    #Check for differences in response distribution by country of residence (just US vs. non-US in this case)
    country_response_distribution = MAP_questionnaire[[country_col] + questions].copy()
    #Create new colum of US vs. non-US and drop original country column
    country_response_distribution['Country_Group'] = country_response_distribution[country_col].apply(lambda x: 'US' if x == 'United States' else 'Non-US')
    country_response_distribution = country_response_distribution.drop(columns=[country_col])

    #Separate data for US and non-US respondents
    us_data = country_response_distribution[country_response_distribution['Country_Group'] == 'US'][questions]
    nonUS_data = country_response_distribution[country_response_distribution['Country_Group'] == 'Non-US'][questions]

    n_us = us_data.shape[0]
    n_nonUS = nonUS_data.shape[0]

    # Dimension-level analysis: Chi-square test of homogeneity for the entire dimension to see if the overall response pattern differs by country group

    # Create contingency table for the entire dimension
    us_counts = us_data.sum().values
    non_us_counts = nonUS_data.sum().values
    
    dimension_contingency_table = np.array([us_counts, non_us_counts])

    chi2_dim, p_chi2_dim, dof_dim, ex_dim = chi2_contingency(dimension_contingency_table)

    #Add dimension-level chi-square results to dimension comparison dataframe
    dimension_comparison.append({
        'Dimension': dimension,
        'Chi2_Statistic': chi2_dim,
        'Chi2_p_value': p_chi2_dim,
        'Degrees_of_Freedom': dof_dim,
        'Comparison_Group': f'Country of Residence (US [{n_us}] vs. Non-US [{n_nonUS}])'
    })

del country_col, country_means_t, chi2_dim, p_chi2_dim, dof_dim, ex_dim, us_data, nonUS_data, us_counts, non_us_counts, country_response_distribution, dimension_contingency_table, n_us, n_nonUS

2. Differences in the selection frequencies across job positions (Controller vs. Non-Controller)

In [50]:
job_means_t = None
job_position_col = "GQ04_2"
#Iterate over each dimension and perform analysis of differences by job position
for dimension, questions in dimension_mapping.items():    
    #Check for differences in response distribution by country of residence (just US vs. non-US in this case)
    job_response_distribution = MAP_questionnaire[[job_position_col] + questions].copy()
    #Create new colum of controller vs. non-controller
    job_response_distribution['Job_Position_Group'] = job_response_distribution[job_position_col].apply(lambda x: 'Controller' if x==4 else 'Non-Controller')
    job_response_distribution = job_response_distribution.drop(columns=[job_position_col])

    #Separate data for controller and non-controller respondents
    controller_data = job_response_distribution[job_response_distribution['Job_Position_Group'] == 'Controller'][questions]
    non_controller_data = job_response_distribution[job_response_distribution['Job_Position_Group'] == 'Non-Controller'][questions]

    n_controller = controller_data.shape[0]
    n_non_controller = non_controller_data.shape[0]

    # Dimension-level analysis: Chi-square test of homogeneity for the entire dimension to see if the overall response pattern differs by job position group
    # Create contingency table for the entire dimension
    controller_counts = controller_data.sum().values
    non_controller_counts = non_controller_data.sum().values
    dimension_contingency_table = np.array([controller_counts, non_controller_counts])
    # Perform chi-square test
    chi2_dim, p_chi2_dim, dof_dim, ex_dim = chi2_contingency(dimension_contingency_table)
    #Add dimension-level chi-square results to dimension comparison dataframe
    dimension_comparison.append({
        'Dimension': dimension,
        'Chi2_Statistic': chi2_dim,
        'Chi2_p_value': p_chi2_dim,
        'Degrees_of_Freedom': dof_dim,
        'Comparison_Group': f'Job Position (Controller [{n_controller}] vs. Non-Controller [{n_non_controller}])'
    })

del job_means_t, job_position_col,chi2_dim, p_chi2_dim, dof_dim, ex_dim, controller_data, non_controller_data, job_response_distribution

3. Differences in the selection frequencies across MA experiences (more/equal than 4 years vs. less)

In [ ]:
experience_means_t = None
experience_col = "GQ06_2"
# Iterate over each dimension and perform analysis of differences by experience
for dimension, questions in dimension_mapping.items():    
    # Check for differences in response distribution by country of residence (just US vs. non-US in this case)
    experience_response_distribution = MAP_questionnaire[[experience_col] + questions].copy()
    # Create new colum of experienced vs. less experienced (4 or more years of experience)
    experience_response_distribution['Experience_Group'] = experience_response_distribution[experience_col].apply(lambda x: 'Experienced' if x >= 3 else 'Less Experienced')
    experience_response_distribution = experience_response_distribution.drop(columns=[experience_col])

    # Separate data for experienced and less experienced respondents
    experienced_data = experience_response_distribution[experience_response_distribution['Experience_Group'] == 'Experienced'][questions]
    less_experienced_data = experience_response_distribution[experience_response_distribution['Experience_Group'] == 'Less Experienced'][questions]

    n_experienced = experienced_data.shape[0]
    n_less_experienced = less_experienced_data.shape[0]

    # Dimension-level analysis: Chi-square test of homogeneity for the entire dimension to see if the overall response pattern differs by experience group
    # Create contingency table for the entire dimension
    experienced_counts = experienced_data.sum().values
    less_experienced_counts = less_experienced_data.sum().values
    dimension_contingency_table = np.array([experienced_counts, less_experienced_counts])
    # Perform chi-square test
    chi2_dim, p_chi2_dim, dof_dim, ex_dim = chi2_contingency(dimension_contingency_table)
    # Add dimension-level chi-square results to dimension comparison dataframe
    dimension_comparison.append({
        'Dimension': dimension,
        'Chi2_Statistic': chi2_dim,
        'Chi2_p_value': p_chi2_dim,
        'Degrees_of_Freedom': dof_dim,
        'Comparison_Group': f'Experience Level (Experienced [{n_experienced}] vs. Less Experienced [{n_less_experienced}])'
    })

# save dimension comparison results to excel
dimension_comparison = pd.DataFrame(dimension_comparison)
dimension_comparison.to_excel('Survey/Tables/Dimension/MAP_Dimension_Comparison_Results.xlsx', index=False)

del experience_means_t, experience_col, chi2_dim, p_chi2_dim, dof_dim, ex_dim, experienced_data, less_experienced_data, experience_response_distribution, dimension_contingency_table, n_experienced, n_less_experienced

<div class="alert-info">
Step 2.3: Final development of MAP dictionary based on selected seed words
</div>

Based on the previous analyses, two approaches for the development of the final MAP dictionary are tested. 

Approach 1: Based on the MAP questionnair, the 5 clusters with the highest selection frequencies (US ranking) are selected (done via Excel spread sheet). Afterwards, we take the word/phrase with the largest number of occurences in the corpus as the final seed word per cluster. In this way, the top 5 seed words for each dimension are selected. The W2V vectors of these 5 words/phrase form the basis for the selection of synonyms.

Approach 2: Based on the MAP questionnair, the 5 clusters with the highest selection frequencies (US ranking) are selected (done via Excel spread sheet). In version two, we now consider all words/phrases in a cluster for the selection of synonyms. To avoid that clusters with more seed words tend to have mechanically more word/phrase occurences, we will calculate the centroid vector of the W2V vectors within each cluster. The resulting centroid vector froms the basis of the selection of seed words.

In [ ]:
# First, we need to load the seed word list which has been created manually using the criterias described above.
seed_words = pd.read_csv('W2V/Dictionary_creation/MAP_Seed_Words_Selection_v1.csv', sep=';')

seed_words_2 = pd.read_csv('W2V/Dictionary_creation/MAP_Seed_Words_Selection_v2.csv', sep=';')

# And we need to load the W2V embeddings 
w2v_file="W2V/Dictionary_creation/w2v_filing_text.model"


w2v = KeyedVectors.load_word2vec_format(w2v_file, binary=True)

In [ ]:
# Approach 1

# Define the MAP dimensions
dimensions = ["Budgeting/Planning", "Cost", "Financing/Investment", "Operations", "Performance/Internal Reporting", "Pricing/Revenue Management", "Risk/Internal Control", "Strategy"]

# Initilize a new dataframe to store the synonyms
synonyms_df = None

# Loop through each dimension and retrieve the synonyms of the top 5 seed words using the W2V embeddings
for dimension in dimensions:
    seed_words_dimension = seed_words[seed_words['Dimension'] == dimension]['Seed_word_final'].tolist()
    for seed_word in seed_words_dimension:
        try:
            similar_words = w2v.most_similar(seed_word, topn=30)
            for word, score in similar_words:
                if synonyms_df is None:
                    synonyms_df = [{'Dimension': dimension, 'Seed_Word': seed_word, 'Synonym': word, 'Similarity_Score': score}]
                else:
                    synonyms_df.append({'Dimension': dimension, 'Seed_Word': seed_word, 'Synonym': word, 'Similarity_Score': score})
        except KeyError:
            print(f"Word '{seed_word}' not found in the Word2Vec model vocabulary.")

# Convert the list of dictionaries to a DataFrame
synonyms_df = pd.DataFrame(synonyms_df)

# Save the synonyms to an Excel file
synonyms_df.to_excel('W2V/Dictionary_creation/MAP_Dimension_Synonyms_W2V_v1.xlsx', index=False)


In [ ]:
# Approach 2

# Initilize a new dataframe to store the synonyms
synonyms_df = None

# Loop through each dimension and retrieve the synonyms of the centroid embedding vector of the top 5 word clusters using the W2V embeddings
for dimension in dimensions:

    # Loop through each cluster in the dimension and get the seed words
    for cluster_num in range(1,6):
        seed_words_cluster = seed_words_2[(seed_words_2['Dimension'] == dimension) & (seed_words_2['Ranking_US'] == cluster_num)]['Seed_word_final'].tolist()
        
        # Get the embeddings for the seed words
        seed_word_embeddings = []
        for seed_word in seed_words_cluster:
            try:
                embedding = w2v[seed_word]
                seed_word_embeddings.append(embedding)
            except KeyError:
                print(f"Word '{seed_word}' not found in the Word2Vec model vocabulary.")
        
        if seed_word_embeddings:
            # Calculate the centroid embedding vector
            centroid_embedding = np.mean(seed_word_embeddings, axis=0)
            
            # Find the most similar words to the centroid embedding
            similar_words = w2v.similar_by_vector(centroid_embedding, topn=30)
            
            for word, score in similar_words:
                if synonyms_df is None:
                    synonyms_df = [{'Dimension': dimension, 'Cluster': cluster_num, 'Seed_words': seed_words_cluster, 'Synonym': word, 'Similarity_Score': score}]
                else:
                    synonyms_df.append({'Dimension': dimension, 'Cluster': cluster_num, 'Seed_words': seed_words_cluster, 'Synonym': word, 'Similarity_Score': score})

# Convert the list of dictionaries to a DataFrame
synonyms_df = pd.DataFrame(synonyms_df)

# Save the synonyms to an Excel file
synonyms_df.to_excel('W2V/Dictionary_creation/MAP_Dimension_Synonyms_W2V_v2.xlsx', index=False) 

Afterwards, the synonyms were manually screened. Duplicate entries present in two dimensions were assigned to the dimension that appeared content-wise to be the most logical fit. Whereas, Synonyms that were not aligned with a particular dimension's content were either discarded or reallocated to a more appropriate dimension, where contextual fit was verified by two independent professionals. The resulting dictionaries are located in the folder "data\W2V\Dictionary_creation" with the file names "MAP_Dictionary_W2V_v1_final.csv" (Approach 1) and "MAP_Dictionary_W2V_v2_final.csv" (Approach 2)